# Frozen-feature linear probe

保存済みの CheXpert 特徴から線形 probe を学習し、全体・属性群別 AUROC を確認する。特徴抽出の条件は変えず、ここでは encoder、属性、正則化強度の候補だけを明示する。

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from projects.foundation_linear_probe.chexpert import add_group_columns, load_split

PROJECT_DIR = Path.cwd() / "projects" / "foundation_linear_probe"
ENCODER = "dinov2_base"
ATTRIBUTES = ("sex", "race", "ethnicity", "insurance_type", "age")
C_GRID = (1e-3, 1e-2, 1e-1, 1.0, 10.0)
FEATURE_DIR = PROJECT_DIR / "outputs" / "features" / ENCODER
RESULT_DIR = PROJECT_DIR / "results" / "probe" / ENCODER


def load_features(path: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """NPZ から image、target、float32 embedding を読み込む。"""
    with np.load(path, allow_pickle=False) as data:
        return data["image"], data["target"], data["embedding"].astype(np.float32)


def fit_probe(
    train: tuple[np.ndarray, np.ndarray, np.ndarray],
    val: tuple[np.ndarray, np.ndarray, np.ndarray],
) -> tuple[object, float, float]:
    """val AUROC が最大の L2 logistic regression と条件を返す。"""
    _, train_target, train_embedding = train
    _, val_target, val_embedding = val
    best_model, best_c, best_auroc = None, None, -np.inf
    for c_value in C_GRID:
        model = make_pipeline(StandardScaler(), LogisticRegression(C=c_value, solver="lbfgs", max_iter=2_000))
        model.fit(train_embedding, train_target)
        auroc = roc_auc_score(val_target, model.predict_proba(val_embedding)[:, 1])
        if auroc > best_auroc:
            best_model, best_c, best_auroc = model, c_value, auroc
    assert best_model is not None and best_c is not None
    return best_model, best_c, best_auroc


def group_auroc(frame: pd.DataFrame, score: np.ndarray) -> pd.DataFrame:
    """全体と、陽性・陰性を含む属性群の AUROC を集計する。"""
    rows = [{"attribute": "overall", "group": "all", "n": len(frame), "auroc": roc_auc_score(frame.target, score)}]
    for attribute in ATTRIBUTES:
        for group, indices in frame.groupby(f"{attribute}_group", observed=True).groups.items():
            target = frame.target.iloc[list(indices)]
            if target.nunique() == 2:
                rows.append(
                    {
                        "attribute": attribute,
                        "group": str(group),
                        "n": len(indices),
                        "auroc": roc_auc_score(target, score[list(indices)]),
                    }
                )
    return pd.DataFrame(rows)

In [ ]:
# Run the selected encoder. Features must have been extracted for all three splits.
train = load_features(FEATURE_DIR / "train.npz")
val = load_features(FEATURE_DIR / "val.npz")
test_images, test_target, test_embedding = load_features(FEATURE_DIR / "test.npz")

model, selected_c, val_auroc = fit_probe(train, val)
test_score = model.predict_proba(test_embedding)[:, 1]

test_frame = load_split("test").set_index("image").loc[test_images].reset_index()
if not np.array_equal(test_frame.target.to_numpy(), test_target):
    raise ValueError("test feature labels do not match the current split")
test_frame = add_group_columns(test_frame, ATTRIBUTES)

metrics = group_auroc(test_frame, test_score)
predictions = test_frame[["image", "target", *[f"{name}_group" for name in ATTRIBUTES]]].copy()
predictions["score"] = test_score

RESULT_DIR.mkdir(parents=True, exist_ok=True)
metrics.to_csv(RESULT_DIR / "group_auroc.csv", index=False)
predictions.to_csv(RESULT_DIR / "test_predictions.csv", index=False)
(RESULT_DIR / "summary.json").write_text(
    json.dumps(
        {
            "encoder": ENCODER,
            "selected_c": selected_c,
            "val_auroc": val_auroc,
            "test_auroc": roc_auc_score(test_target, test_score),
        },
        indent=2,
    )
    + "\n"
)

display(metrics)